# Bloque II — Regresión y comparación de modelos
## Notebook Entregable — Ejercicio Integrador
**Duración estimada:** 3 horas  
**Dataset:** `../data/ventas_mayo_2026.csv`

---

## Objetivo de aprendizaje

El alumnado aprenderá a construir modelos de regresión para predecir una variable numérica continua, comparar distintos algoritmos y justificar la elección del modelo mediante métricas.

---

## Agenda de 3 horas

| Tiempo | Actividad |
|--------|----------|
| 0:00 – 0:25 | Qué es un problema de regresión |
| 0:25 – 0:55 | Preparación del dataset |
| 0:55 – 1:25 | Regresión lineal simple y múltiple |
| 1:25 – 1:35 | ☕ Pausa |
| 1:35 – 2:05 | Ridge, Lasso y Random Forest |
| 2:05 – 2:35 | Métricas y comparación |
| 2:35 – 3:00 | Caso práctico final |

---
## 0. Configuración del entorno

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

print('Librerías cargadas correctamente ✓')

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('sklearn cargado correctamente ✓')

---
## 1. ¿Qué es un problema de regresión?

En **regresión supervisada** intentamos predecir una **variable continua** (numérica) a partir de un conjunto de variables predictoras.

### Ejemplos del mundo real
| Contexto | Variable objetivo |
|----------|------------------|
| Inmobiliario | Precio de venta de un piso |
| Ventas | Importe de una transacción |
| RRHH | Salario estimado de un candidato |
| Energía | Consumo eléctrico en kWh |

### Diferencia con clasificación
- **Clasificación** → salida discreta: *¿A qué clase pertenece?*
- **Regresión** → salida continua: *¿Cuánto vale?*

### Fórmula general (regresión lineal)
$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \ldots + \beta_p x_p + \varepsilon$$

El modelo **aprende** los coeficientes $\beta$ minimizando el error cuadrático sobre los datos de entrenamiento.

---
## 2. Carga y preparación inicial del dataset

In [ ]:
df = pd.read_csv('../data/ventas_mayo_2026.csv')
df = df.drop_duplicates()
df['fecha'] = pd.to_datetime(df['fecha'])
df['mes'] = df['fecha'].dt.month

print(f'Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}')
df.head()

In [ ]:
# Estadísticas descriptivas
df.describe()

In [ ]:
# Distribución de la variable objetivo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['importe'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución del importe')
axes[0].set_xlabel('Importe (€)')
axes[0].set_ylabel('Frecuencia')

df.groupby('categoria')['importe'].mean().sort_values().plot(
    kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Importe medio por categoría')
axes[1].set_xlabel('Importe medio (€)')

plt.tight_layout()
plt.show()

---
## 3. Definición de X e y

- **`X`** — variables predictoras (features)
- **`y`** — variable objetivo que queremos predecir: `importe`

In [ ]:
TARGET = 'importe'

features_num = ['unidades', 'precio_unitario', 'descuento',
                'antiguedad_cliente_meses', 'mes']
features_cat = ['categoria', 'region', 'canal']

X = df[features_num + features_cat]
y = df[TARGET]

print(f'X: {X.shape}  |  y: {y.shape}')
X.head()

---
## 4. División train / test

> **Regla fundamental:** el modelo **nunca debe ver los datos de test** durante el entrenamiento.  
> Si lo hace, las métricas estarán infladas y no serán representativas del comportamiento real.

Usamos **80 % entrenamiento / 20 % test** con `random_state=42` para reproducibilidad.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {X_train.shape[0]:,} filas | Test: {X_test.shape[0]:,} filas')

---
## 5. Preprocesamiento

Los algoritmos de ML necesitan entrada **numérica**. Construimos un pipeline de preprocesamiento que:

1. **Imputa** valores nulos (mediana para numéricas, moda para categóricas)
2. **Escala** las numéricas con `StandardScaler` → media 0, desviación 1
3. **Codifica** las categóricas con `OneHotEncoder`

Usar `Pipeline` evita **data leakage**: el scaler aprende solo con train y se aplica a test.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, features_num),
        ('cat', categorical_transformer, features_cat)
    ]
)

print('Preprocesador definido ✓')

---
## 6. Función de evaluación unificada

| Métrica | Descripción | ¿Mejor cuando...? |
|---------|-------------|-------------------|
| **MAE** | Error absoluto medio | Queremos interpretar el error en euros |
| **RMSE** | Raíz del error cuadrático medio | Penalizamos errores grandes |
| **R²** | Variabilidad explicada (0–1) | Queremos comparar modelos |

> **RMSE vs MAE:** si un error de 500 € es mucho peor que cinco errores de 100 €, usa RMSE.

In [ ]:
def evaluar_modelo(nombre, modelo, X_train, X_test, y_train, y_test):
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)

    mae  = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2   = r2_score(y_test, pred)

    print(f'[{nombre}]  MAE={mae:,.1f} €  |  RMSE={rmse:,.1f} €  |  R²={r2:.4f}')
    return {'modelo': nombre, 'MAE': mae, 'RMSE': rmse, 'R2': r2}, pred

---
## 7. Regresión lineal

**Por qué empezar aquí:**
- Rápida de entrenar e interpretar
- Establece la **línea base** (*baseline*) que los modelos más complejos deben superar
- Los coeficientes explican el impacto de cada variable

**Limitaciones:**
- Asume relaciones lineales
- Sensible a *outliers* y a la multicolinealidad

In [ ]:
modelo_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

res_lr, pred_lr = evaluar_modelo(
    'Linear Regression', modelo_lr,
    X_train, X_test, y_train, y_test
)

In [ ]:
# Visualizamos los coeficientes del modelo lineal
ohe_cats = (modelo_lr
            .named_steps['preprocessor']
            .named_transformers_['cat']
            .named_steps['onehot']
            .get_feature_names_out(features_cat))

feature_names = features_num + list(ohe_cats)
coefs = modelo_lr.named_steps['model'].coef_

coef_df = pd.DataFrame({'feature': feature_names, 'coef': coefs})
coef_df = coef_df.reindex(coef_df['coef'].abs().sort_values(ascending=False).index)

plt.figure(figsize=(9, 5))
colors = ['steelblue' if c > 0 else 'coral' for c in coef_df['coef']]
plt.barh(coef_df['feature'], coef_df['coef'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Coeficientes — Regresión Lineal')
plt.xlabel('Valor del coeficiente (escala normalizada)')
plt.tight_layout()
plt.show()

---
## 8. Ridge y Lasso — Regularización

Cuando los coeficientes crecen demasiado el modelo **sobreajusta** (memoriza el train pero falla en test). La regularización añade una penalización que frena ese crecimiento.

| Modelo | Penalización | Efecto |
|--------|-------------|--------|
| **Ridge** (L2) | suma de cuadrados de $\beta$ | Reduce coeficientes, los mantiene todos |
| **Lasso** (L1) | suma de valores absolutos de $\beta$ | Puede llevar coeficientes a **cero** (selección automática) |

> El hiperparámetro `alpha` controla la intensidad: mayor `alpha` → más regularización.

In [ ]:
modelo_ridge = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Ridge(alpha=1.0))
])

modelo_lasso = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Lasso(alpha=0.05, max_iter=10000))
])

res_ridge, pred_ridge = evaluar_modelo(
    'Ridge', modelo_ridge, X_train, X_test, y_train, y_test)
res_lasso, pred_lasso = evaluar_modelo(
    'Lasso', modelo_lasso, X_train, X_test, y_train, y_test)

In [ ]:
# Comparación de alphas en Ridge
alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
rmse_alphas = []

for a in alphas:
    m = Pipeline(steps=[('preprocessor', preprocessor), ('model', Ridge(alpha=a))])
    m.fit(X_train, y_train)
    p = m.predict(X_test)
    rmse_alphas.append(np.sqrt(mean_squared_error(y_test, p)))

plt.figure(figsize=(7, 4))
plt.semilogx(alphas, rmse_alphas, marker='o', color='steelblue')
plt.title('Ridge — RMSE según alpha')
plt.xlabel('Alpha (escala logarítmica)')
plt.ylabel('RMSE (€)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 9. Random Forest Regressor

Un **Random Forest** construye cientos de árboles de decisión sobre subconjuntos aleatorios de datos y variables, y promedia sus predicciones.

**Ventajas:**
- Captura relaciones **no lineales** e interacciones
- Robusto ante outliers
- Proporciona importancia de variables

**Inconvenientes:**
- Menos interpretable
- Más lento de entrenar y desplegar
- Puede sobreajustar sin control (`max_depth`)

In [ ]:
modelo_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        max_depth=None
    ))
])

res_rf, pred_rf = evaluar_modelo(
    'Random Forest', modelo_rf, X_train, X_test, y_train, y_test)

In [ ]:
# Importancia de variables — Random Forest
ohe_cats_rf = (modelo_rf
               .named_steps['preprocessor']
               .named_transformers_['cat']
               .named_steps['onehot']
               .get_feature_names_out(features_cat))

feature_names_rf = features_num + list(ohe_cats_rf)
importances = modelo_rf.named_steps['model'].feature_importances_

imp_df = (pd.DataFrame({'feature': feature_names_rf, 'importance': importances})
            .sort_values('importance', ascending=False)
            .head(12))

plt.figure(figsize=(8, 5))
plt.barh(imp_df['feature'][::-1], imp_df['importance'][::-1], color='seagreen')
plt.title('Importancia de variables — Random Forest')
plt.xlabel('Importancia relativa')
plt.tight_layout()
plt.show()

---
## 10. Comparación de modelos

In [ ]:
resultados = pd.DataFrame([res_lr, res_ridge, res_lasso, res_rf])
resultados = resultados.sort_values('RMSE').reset_index(drop=True)
resultados

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
modelos = resultados['modelo']
colores = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

for ax, metrica, titulo in zip(
    axes,
    ['RMSE', 'MAE', 'R2'],
    ['RMSE (€) — menor es mejor',
     'MAE (€) — menor es mejor',
     'R² — mayor es mejor']
):
    bars = ax.bar(modelos, resultados[metrica], color=colores)
    ax.set_title(titulo, fontsize=10)
    ax.tick_params(axis='x', rotation=20)
    for bar, val in zip(bars, resultados[metrica]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() * 0.5,
                f'{val:,.2f}',
                ha='center', va='center',
                color='white', fontweight='bold', fontsize=9)

plt.suptitle('Comparación de modelos de regresión', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 11. Análisis de residuos y gráfico real vs predicho

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Real vs predicho
axes[0].scatter(y_test, pred_rf, alpha=0.5, color='steelblue', s=15)
lims = [y_test.min(), y_test.max()]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Predicción perfecta')
axes[0].set_xlabel('Valor real (€)')
axes[0].set_ylabel('Predicción (€)')
axes[0].set_title('Real vs Predicho — Random Forest')
axes[0].legend()

# Residuos
residuos = y_test.values - pred_rf
axes[1].scatter(pred_rf, residuos, alpha=0.5, color='coral', s=15)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xlabel('Predicción (€)')
axes[1].set_ylabel('Residuo (real - predicho)')
axes[1].set_title('Análisis de residuos — Random Forest')

plt.tight_layout()
plt.show()

print(f'Residuo medio: {residuos.mean():,.2f} €')
print(f'Residuo std:   {residuos.std():,.2f} €')

---
## 12. Ejercicio integrador ✏️

Trabaja con los siguientes puntos y anota tus conclusiones en las celdas de abajo.

1. **Cambia el porcentaje de test a 30 %** y compara los resultados con el 20 %.
2. **Modifica `max_depth`** en Random Forest (prueba 5, 10, 20) y observa el efecto.
3. **Añade o elimina variables** predictoras del conjunto de features.
4. **Justifica qué modelo seleccionarías** y por qué.
5. **Redacta una conclusión de negocio** (2-3 frases).

In [ ]:
# --- EJERCICIO 1: Cambia test_size a 0.30 ---
X_train_30, X_test_30, y_train_30, y_test_30 = train_test_split(
    X, y, test_size=0.30, random_state=42
)

res_rf_30, _ = evaluar_modelo(
    'RF test=30%', modelo_rf, X_train_30, X_test_30, y_train_30, y_test_30)

print('\nComparación 20 % vs 30 % de test:')
pd.DataFrame([res_rf, res_rf_30])

In [ ]:
# --- EJERCICIO 2: Efecto de max_depth ---
resultados_depth = []
for depth in [5, 10, 20, None]:
    m = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', RandomForestRegressor(n_estimators=100, max_depth=depth, random_state=42))
    ])
    label = f'RF depth={depth}' if depth else 'RF depth=None'
    r, _ = evaluar_modelo(label, m, X_train, X_test, y_train, y_test)
    resultados_depth.append(r)

pd.DataFrame(resultados_depth).sort_values('RMSE')

In [ ]:
# --- EJERCICIO 3: Añadir / eliminar variables ---
# Prueba quitando 'mes' o 'region' para ver el impacto
features_reducido = ['unidades', 'precio_unitario', 'descuento']
features_cat_red  = ['categoria', 'canal']

X_red = df[features_reducido + features_cat_red]
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_red, y, test_size=0.2, random_state=42)

pre_red = ColumnTransformer(transformers=[
    ('num', numeric_transformer, features_reducido),
    ('cat', categorical_transformer, features_cat_red)
])

m_red = Pipeline(steps=[
    ('preprocessor', pre_red),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

res_red, _ = evaluar_modelo('RF features reducidas', m_red,
                             X_train_r, X_test_r, y_train_r, y_test_r)

print('\n--- Comparación features completas vs reducidas ---')
pd.DataFrame([res_rf, res_red])

### 📝 Tus conclusiones

**Modelo seleccionado:**  
*Escribe aquí tu elección y justificación...*

**Conclusión de negocio:**  
*Escribe aquí 2-3 frases con el impacto práctico del modelo...*

---
## 13. Pregunta de cierre

> ¿Elegirías **siempre** el modelo con menor RMSE? Justifica tu respuesta.

*(Reflexiona antes de leer la respuesta en `docs/respuesta_pregunta_cierre.md`)*

---
## Referencias

- Scikit-learn docs: https://scikit-learn.org/stable/
- Hastie, Tibshirani & Friedman — *The Elements of Statistical Learning*
- James et al. — *An Introduction to Statistical Learning*